# Fine-tuned Sneaker Top-5 Inference

Upload a fine-tuned checkpoint and an image, then get the top-5 predicted sneaker labels.

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git pillow matplotlib ftfy regex tqdm

In [ ]:
from pathlib import Path

import clip
import matplotlib.pyplot as plt
import torch
from google.colab import files
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B/32"


def format_class_name(class_name: str) -> str:
    return class_name.replace("_", " ").title()


def load_classifier(checkpoint_path: str | Path):
    model, preprocess = clip.load(MODEL_NAME, device=DEVICE, jit=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    class_names = list(checkpoint["class_names"])
    class_prompts = [f"a photo of {format_class_name(name)} sneakers" for name in class_names]
    text_tokens = clip.tokenize(class_prompts).to(DEVICE)

    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    return {
        "model": model,
        "preprocess": preprocess,
        "class_names": class_names,
        "class_prompts": class_prompts,
        "text_features": text_features,
    }


@torch.no_grad()
def predict_top_k(classifier, image_path: str | Path, k: int = 5):
    image = Image.open(image_path).convert("RGB")
    image_tensor = classifier["preprocess"](image).unsqueeze(0).to(DEVICE)
    image_features = classifier["model"].encode_image(image_tensor)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    probabilities = (100.0 * image_features @ classifier["text_features"].T).softmax(dim=-1)[0]

    top_scores, top_indices = probabilities.topk(min(k, len(classifier["class_names"])))
    rows = []
    for score, index in zip(top_scores.tolist(), top_indices.tolist()):
        rows.append({
            "class_name": classifier["class_names"][index],
            "label": format_class_name(classifier["class_names"][index]),
            "prompt": classifier["class_prompts"][index],
            "score": float(score),
        })

    return image, rows

In [ ]:
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

After upload, set `CHECKPOINT_FILE` to your `.pt` file and `IMAGE_FILE` to your test image.

In [ ]:
CHECKPOINT_FILE = "clip_sneaker_best.pt"
IMAGE_FILE = "example.jpg"
TOP_K = 5

classifier = load_classifier(CHECKPOINT_FILE)
image, predictions = predict_top_k(classifier, IMAGE_FILE, k=TOP_K)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.show()

for rank, row in enumerate(predictions, start=1):
    print(f"{rank}. {row['label']} ({row['score']:.4f})")
    print(f"   class_name={row['class_name']}")
    print(f"   prompt={row['prompt']}")